# FlexiSLM Minimal Inference Notebook
Minimal example for importing `src.inference_flexislm` and running T2T/S2T/TTS.
`auto_download=True` fetches checkpoints with `snapshot_download` into the repo `models/` folder (same layout as manual `hf download`).


In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd().resolve()
while not (repo_root / "src").is_dir():
    if repo_root.parent == repo_root:
        raise FileNotFoundError("Could not find repo root (parent of src/)")
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.inference_flexislm import FlexiSLMInferenceConfig, FlexiSLMInference

# Automatic: snapshot_download into the repo models/ folder (same as manual hf download).
# For local paths instead, set auto_download=False and pass model_path, encoder, FlexiCodec, and SenseVoice.
cfg = FlexiSLMInferenceConfig(
    auto_download=True,
    use_flow_matching_decoder=False,
    enable_flexible_framerate=True,
    input_framerate=8.0,
    default_framerate=8.0,
)
engine = FlexiSLMInference(cfg, device='cuda')

# T2T
t2t = engine.generate_from_text(
    text_input='Explain dynamic frame rate in one sentence.',
    history='',
    framerate=1.0,
    output_text_only=True,
)
print('[T2T]', t2t.get('text', ''))

# S2T
s2t = engine.generate_from_audio(
    audio_path='path/to/input_audio.wav',
    text_query='Please transcribe the speech.',
    history='',
    framerate=1.0,
    output_text_only=True,
)
print('[S2T]', s2t.get('text', ''))

# TTS
tts = engine.generate_tts(
    sentence='Hello, this is a minimal FlexiSLM notebook example.',
    framerate=1.0,
)
print('[TTS text]', tts.get('text', ''))
print('[TTS audio tokens]', None if tts.get('audio_ids') is None else len(tts['audio_ids']))
